In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import re


result_folder = "decentralized_simulation"

df = pd.read_csv(f"~/Reasoning/Decentralized-Agents/experiments/results/{result_folder}/duel_record.csv")
df = df.sort_values("timestamp")
t0 = df["timestamp"].iloc[0]
df["rel_time"] = df["timestamp"] - t0

def parse_node_amount(s: str):
    result = {}

    if pd.isna(s):
        return result
    s = s.strip()
    if not s:
        return result

    items = s.split(";")
    for item in items:
        item = item.strip()
        if not item:
            continue
        if ":" not in item:
            continue  # ignore malformed
        node, amt = item.split(":", 1)
        node = node.strip()
        amt = amt.strip()
        try:
            result[node] = float(amt)
        except:
            pass  # skip invalid numbers

    return result


duel_records = []
reward_records = []

for _, row in df.iterrows():
    ts = row["rel_time"]

    dec = parse_node_amount(row["decrease"])
    inc = parse_node_amount(row["increase"])

    if row["action"] == "reward":
        for node, amt in inc.items():
            reward_records.append({"rel_time": ts, "node": node, "reward": amt})
        for node, amt in dec.items():
            reward_records.append({"rel_time": ts, "node": node, "reward": -amt})
        continue

    for node, amt in dec.items():
        duel_records.append({"rel_time": ts, "node": node, "delta": -amt})

    for node, amt in inc.items():
        duel_records.append({"rel_time": ts, "node": node, "delta": amt})

df_delta = pd.DataFrame(duel_records)
df_reward = pd.DataFrame(reward_records)


df_trend = df_delta.pivot_table(
    index="rel_time",
    columns="node",
    values="delta",
    aggfunc="sum",
    fill_value=0,
)
df_cum = df_trend.cumsum()

if not df_reward.empty:
    df_reward_trend = df_reward.pivot_table(
        index="rel_time",
        columns="node",
        values="reward",
        aggfunc="sum",
        fill_value=0,
    )
    df_reward_cum = df_reward_trend.cumsum()
else:
    df_reward_cum = pd.DataFrame()

def extract_node_id(name):
    m = re.search(r"(\d+)$", name)
    return int(m.group(1)) if m else float("inf")


all_nodes = sorted(
    list(set(df_cum.columns.tolist()) | set(df_reward_cum.columns.tolist())),
    key=extract_node_id
)

cmap = plt.cm.get_cmap("tab20", len(all_nodes))
node_color = {node: cmap(i) for i, node in enumerate(all_nodes)}


fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax_duel, ax_reward = axes

legend_handles = []
legend_labels = []


# =====================================================
# ===============   Left: duel cumulative   ============
# =====================================================
nodes_duel = sorted(df_cum.columns.tolist(), key=extract_node_id)

# pair nodes two-by-two
pairs_duel = []
i = 0
while i < len(nodes_duel):
    if i + 1 < len(nodes_duel):
        pairs_duel.append([nodes_duel[i], nodes_duel[i+1]])
    else:
        pairs_duel.append([nodes_duel[i]])
    i += 2

# raw duel curves
for n in nodes_duel:
    ax_duel.plot(df_cum.index, df_cum[n],
                 color=node_color[n],
                 alpha=0.25,
                 linewidth=1)

for pair in pairs_duel:
    if len(pair) == 1:
        avg = df_cum[pair[0]]
        color = node_color[pair[0]]
        label = f"{pair[0]}"
    else:
        avg = df_cum[pair].mean(axis=1)
        color = node_color[pair[0]]
        label = f"{pair[0]}-{pair[1]}"

    line, = ax_duel.plot(df_cum.index, avg,
                         linewidth=2,
                         color=color)

ax_duel.set_title("Duel Cumulative Credit")
ax_duel.set_xlabel("Relative Time (s)")
ax_duel.set_ylabel("Cumulative Credit")
ax_duel.grid(True)


# =====================================================
# ===============   Right: reward cumulative ==========
# =====================================================
nodes_reward = sorted(df_reward_cum.columns.tolist(), key=extract_node_id)
nodes_reward = [n for n in nodes_reward if n != "node0"]

# pair the remaining reward nodes
pairs_reward = []
i = 0
while i < len(nodes_reward):
    if i + 1 < len(nodes_reward):
        pairs_reward.append([nodes_reward[i], nodes_reward[i+1]])
    else:
        pairs_reward.append([nodes_reward[i]])
    i += 2

# raw reward curves
for n in nodes_reward:
    ax_reward.plot(df_reward_cum.index, df_reward_cum[n],
                   alpha=0.25,
                   linewidth=1,
                   color=node_color[n])

# pair averages
for pair in pairs_reward:
    if len(pair) == 1:
        avg = df_reward_cum[pair[0]]
        color = node_color[pair[0]]
        label = f"{pair[0]}"
    else:
        avg = df_reward_cum[pair].mean(axis=1)
        color = node_color[pair[0]]
        label = f"{pair[0]}-{pair[1]}"

    line, = ax_reward.plot(df_reward_cum.index, avg,
                           linewidth=2,
                           color=color)
    legend_handles.append(line)
    legend_labels.append(label)

ax_reward.set_title("Reward Cumulative Credit")
ax_reward.set_xlabel("Relative Time (s)")
ax_reward.grid(True)


# =====================================================
# ===============   Shared Legend   ===================
# =====================================================
fig.legend(
    handles=legend_handles,
    labels=legend_labels,
    loc='upper center',
    ncol=6,
    frameon=False,
    fontsize=9,
    bbox_to_anchor=(0.5, 1.08)
)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()